# Загрузка, анализ, подготовка данных

## Задача

После разбора домашнего задания выяснелось, что данные были зашумлены. Вот некоторые из признаков проблемных переводов:
1) Длинна перевода больше длинны оргинала более чем в $n$ раз (и наоборот) Где $n$ определяется эмперически для каждого датасета;
2) В паре примера одно из полей (`src` и/или `dst`) пустое;
3) В переводе и оригинале используются арабские цифры, но они различаются;
4) в паре используются скобки и кавычки, они различаются.

## Подготовка среды

### Импорт библиотек

In [71]:
import random
import string
import re

import pandas as pd
import numpy as np
import plotly.express as px

from datasets import load_dataset, Features, Value, concatenate_datasets, Dataset, DatasetDict

### Полезные функции

In [72]:
# Функция для подсчета уникальных символов
def count_unique_characters_in_split(dataset, split_name, feature_name):
    
    unique_characters = set()
    
    for example in dataset[split_name][feature_name]:
        if example is not None:
            unique_characters.update(example)
    num_unique_characters = len(unique_characters)

    print(f"Number of unique characters in {split_name} {feature_name} split: {num_unique_characters}")
    print(f"Unique characters in {split_name} {feature_name} split: {unique_characters}", end="\n\n")

    return num_unique_characters, unique_characters

# Функция для подсчета отношения длин src и dst и рисования гистограммы
def plot_length_difference_distribution(df, split_name='train', bins=100):
    df['length_difference'] = df['src'].apply(len) / df['dst'].apply(len)
    fig = px.histogram(df, x='length_difference', nbins=bins, title=f'Распределение разницы в длине между src и dst для {split_name}')
    
    tick_step = 0.25
    x_range = [0, 3.5]
    tickvals = np.arange(x_range[0], x_range[1] + tick_step, tick_step)
    
    fig.update_xaxes(title='Отношение длин', range=x_range, tickvals=tickvals)

    fig.show()

## Загрузка данных

Загрузим данные и посмотрим на случайно выбранный примеры из каждой части датасета:

In [73]:
data_files = {
    "train": "raw_train.jsonl",
    "validation": "raw_val.jsonl",
    "test": "raw_test_no_reference.jsonl"
}

features = Features({
    'src': Value('string'),
    'dst': Value('string'),
})

dataset = load_dataset("json", data_files=data_files, features=features)
dataset

DatasetDict({
    train: Dataset({
        features: ['src', 'dst'],
        num_rows: 300000
    })
    validation: Dataset({
        features: ['src', 'dst'],
        num_rows: 500
    })
    test: Dataset({
        features: ['src', 'dst'],
        num_rows: 1000
    })
})

In [74]:
example_index = random.randint(0, dataset['validation'].shape[0] - 1)

print(f"Train example: {dataset['train'][example_index]}, src lengh {len(dataset['train'][example_index]['src'])}, dst lengh {len(dataset['train'][example_index]['dst'])}")
print(f"Validation example: {dataset['validation'][example_index]}, src lengh {len(dataset['validation'][example_index]['src'])}, dst lengh {len(dataset['validation'][example_index]['dst'])}")
print(f"Test example: {dataset['test'][example_index]}, src lengh {len(dataset['test'][example_index]['src'])}")

Train example: {'src': '▮◠◈◪▼▴ ▫◠▨▪▱▪◳◧◓◈▾▨▵', 'dst': "We're only kidding."}, src lengh 20, dst lengh 19
Validation example: {'src': "◰◞▨◫ ◀◠◒▨◠▦ ◚▴ ◝◂▱◫◚◳◠'▦▪▦ ▲◠▷▴◳'◈▴▨◗ ▫◪◎◞◗▱▼◗◞◗ ◘◈▾◠◓◈◂ ▤◂◈◓í◕◨▴▢ ◑▴▱▫▢é■ ◎◠▷▨◪◎◪▦◫▦ ▨◠◓◠◓▪ ◧▱◠◐◠▦ ◈◬◒▪ ◀◗◓ ▷◬▢▱◠ ◠▱◈▪◐◬▦◠ ◈◠◗◓ ◗◈◈◗◠▱◠◓▪ ◓▴◈◈▴▫▫◫▵", 'dst': "Former President Eduardo Rodriguez Velce, Bolivia's representative in The Hague, disagrees that the court's decision was made too quickly."}, src lengh 148, dst lengh 138
Test example: {'src': "◟▴▫▨◗▱◫▱▴◓■ ▼▾◎◠ ◕▩▦◭ ◒◫◈◈◪▫▱◫ ◀◫◓ ◈▴▻◓▴◎ ◚◪ ▫◞▾▦◠◎◗ ◞▴◀◪◀◗▽▱◪ ◰▦◈◧▦▴▢◳◠'▦▪▦ □◠▱◨ ◒▴▷◓◗▦◈◪ ◪▦ ◠▢ 384 ▨◫◒◫▦◫▦ ▷◠◳◠▫▪▦◬ ▨◠▽◀▴▫▫◗◐◫▦◫ ◞◣▽▱▴◈◗▵ ○◳◓◬▼◠■ ◣▱◭ ◞◠▽▪◞▪▦▪▦ ▽▩▨◞◪▱◎▴◞◗ ◀▴▨▱▴▦◫◳◂◓▵", 'dst': None}, src lengh 184


Судя по выводу, данные загрузились без проблем с кодировкой. Посмотрим на используемые во всех частях датасета символы:

In [75]:
# Признак src
train_src_unique_char_count, train_src_unique_chars = count_unique_characters_in_split(dataset, 'train', 'src')
validation_src_unique_char_count, validation_src_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'src')
test_unique_src_char_count, test_src_unique_chars = count_unique_characters_in_split(dataset, 'test', 'src')

# Признак dst
train_dst_unique_char_count, train_dst_unique_chars = count_unique_characters_in_split(dataset, 'train', 'dst')
validation_dst_unique_char_count, validation_dst_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'dst')

# Загрязняющие символы для каждого из признаков
src_chars_to_remove = train_src_unique_chars - (validation_src_unique_chars | test_src_unique_chars)
print(f"SRC feature have {len(src_chars_to_remove)} symbols for remove: {src_chars_to_remove} )")

dst_chars_to_remove = train_dst_unique_chars - validation_dst_unique_chars
print(f"DST feature have {len(dst_chars_to_remove)} symbols for remove: {dst_chars_to_remove} )")

Number of unique characters in train src split: 176
Unique characters in train src split: {'■', 'ý', '▯', 'Ä', '▿', 'Â', '▾', '?', '▮', '▸', '/', '▩', '—', 'ø', '♪', '[', '△', '°', '◊', 'ß', '%', 'ν', 'ú', '◢', '7', '◣', 'û', '◤', '◔', '+', '}', '▲', '▤', '!', 'á', ':', '¤', '◓', '▻', '◂', '◕', 'ñ', '@', 'ă', '▱', '▰', '◨', '{', 'ð', 'É', 'þ', '◝', ';', '◮', '◑', '"', '◛', '▹', '£', '□', '=', '◄', '\x9e', '▼', 'ο', '◌', 'ô', '◰', '◞', '◫', 'ţ', '\u202d', '▥', 'è', '◪', '(', '▬', '◲', 'ι', '\u200b', '▪', '_', '▵', '◁', 'Þ', '◒', '◬', '▨', '◳', 'î', 'º', '◅', '#', '▫', 'ï', 'Ý', '4', '◗', '¡', '•', '`', 'â', '5', '◚', '◩', '*', '◧', 'ó', 'ª', '◭', '2', 'đ', '0', '▣', '●', '▢', '◉', '\x9d', '$', 'í', ' ', '▭', '◟', '◐', '─', '~', '◥', '◙', '◍', '´', 'å', '◜', '○', 'Î', '▦', 'ä', '◯', '3', '8', '™', '▽', '¿', '◎', ']', '◀', '◡', '‚', '^', '1', '◆', '◖', "'", 'ë', '◠', '◈', '▶', '▧', '6', '◇', '▷', '►', '◦', '◃', '\\', '◘', '§', 'é', '◱', '▴', '♫', '\x99', '–', '9', ')', '\xa0', '¶'}

Numbe

## Очистка данных

In [76]:
df_train = pd.DataFrame(dataset['train'])
df_validation = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,=Grasshopper is moving out of the crowd.=
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Строки в которых нет английских букв в признаке `dst`

In [77]:
# Фильтрация строк, в которых в признаке dst нет английских букв
df_train_no_english = df_train[~df_train['dst'].str.contains('[a-zA-Z]', regex=True)]
df_train_no_english

,src,dst
1366,◝◨▦◨ ◞◠▦◠ ▨◗◎ ◞◇◳▱◪◈◗?,_
1739,"""◟○▲○▯▬◃ ◙◘◙◰◢▶◲◌""",_
2257,► ◡◠▻◧▦ ◎◠▱◬▵,"- $130,000."
3601,◤◗◓◎◗ ◀▴◒▵,285...
5646,◂▱◈▾◐▾▦◨ ◈◭◒◭▦◎▩◒▫▩◎▵ ○◎◠ ◈◪◐◗▱◈◫■ ◀▩▽◭▨ ◀◫◓ ◒...,_
...,...,...
294982,◝◗◓◕◪◓■ ▷▴◓ ▢◠◎◠▦ ◠◀◠◓▫▪▱▪ ◈◠◚◓◠▦◬▽◧◓ ▢◠▫◪▦▵,_
295836,# ◆◇▦◈▴◓ ▫◂▻▱◨▱◨◐◨▦ ◗◉◗▦▴ #,_
296309,▮◪▦◫ ▫◪◀◓◫▨ ▴▫◎◪◳◗ ◈◭◒◭▦▩◳◧◓◈◨◎ ◠◎◠ ◀▾ ◕◭◚◪▦◗▦...,_
299240,1400▵ 1400 ◈◂▱◠◓ ◚◪◓◪▦ ◚◠◓ ◎◬? 1400▵,"1,400."


Удалим такие примеры:

In [78]:
df_train = df_train.drop(df_train_no_english.index)
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,=Grasshopper is moving out of the crowd.=
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Вопросительные и восклицательные предложения предложения

Знаки `!` и `?` в обоих языках должны совпадать в конце предложения. Найдём примеры в которых это не соблюдается:

In [79]:
examples_with_exclamation_question_mismatch = df_train[(df_train['src'].str.endswith(('!', '?'))) & ~df_train['dst'].str.endswith(('!', '?'))]
examples_with_exclamation_question_mismatch

,src,dst
69,◢◠▻◠◐◬▦ ◠◉▪▱◎◠◞▪ ◫▱◕◫▦◗ ◉▴▨◎▴◈◫ ◎◫?,No interest in the hatch blowing.
146,◝◗▢◫ ◀▾◓◠▽◠ ▦◪◈▴▦ ◕◪▫◗◓◈◫▦?,There are trash cans.
185,▶◠◎◠◎■ ◚◠◳ ▼◠▦◬▦◠!,Ok. Wow.
228,◝◪▦◫◎▱▴ ◠▱◠▽ ▴◈◗◳◧◓◞◨▦ ▷▴◓▷â▱◈▴?,You've got to be kidding me.
240,◑◪◓ ◒◨▦▾■ ◳◭◓◭ ▷◠◈◗■ ▷◠◓◪▨▴▫ ▴▫!,"Give me that. Go on, get moving."
...,...,...
299811,◤◠▫▫▪◐◬◎▪▢◈◠■ ◀◪▦ ◗▱▨ ◎◗ ◧▱◠▼◠◐▪◎?,"When we bed together, shall I be your first? -..."
299855,◆◣◓▴◚◫ ◀▪◓◠▨◎▪◒ ◂▱◨◓◈◨▨?,We would've dropped the charges.
299879,▭◠▽◈◫ ◂◐▱▾◎!,"Come on, boy."
299980,► ◄◫▨◪ ▶▽◞◧▦'▪ ▨◫◎ ◉◠▱▪◒▫◬◓◈◬ ◞◠▦▪◳◂◓◞▾▦?,- Yeah. - Gus D'Amato.


Удалим такие примеры из тренировочной части датасета:

In [80]:
df_train = df_train.drop(examples_with_exclamation_question_mismatch.index)
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,=Grasshopper is moving out of the crowd.=
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


###  Примеры заканчивающиеся символами `:` в обоих признаках

Посмотрим на примеры которые заканчиваются символом `:` в признаках `src` и не заканчиваются этим символом в признаке `dst` и наоборот:

In [81]:
# Examples where src ends with ':' but dst does not
examples_src = df_train[(df_train['src'].str.endswith(':')) & (~df_train['dst'].str.endswith(':'))]

# Examples where dst ends with ':' but src does not
examples_dst = df_train[(~df_train['src'].str.endswith(':')) & (df_train['src'].str.endswith(':'))]

Удалим из тренировочной части такие примеры:

In [82]:
df_train = df_train.drop(examples_src.index)
df_train = df_train.drop(examples_dst.index)

df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,=Grasshopper is moving out of the crowd.=
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Примеры с проблемами с прямой речью

В тестовой и валидационной частях датасета нет примеров с прямой речью, которые начинаются с символа `►` в признаке `src` но начинаются с символов '-' или '—' в признаке `dst`. Удалим их тренировочной части датасета:

In [83]:
df_train = df_train[~((df_train['src'].str.startswith('►')) | (df_train['dst'].str.startswith(('-', '—'))))]
df_train

,src,dst
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
5,◝▴▦ ◫◒▫◪▦ ▨◠◳▫◠◓◠▼◠▨ ◀◗◓◫ ◈▴◐◫▱◈◗◓▵,"For instance, Ben is not a guy who skips work."
...,...,...
299994,▯▴◓▴▽▴ ◕◫▫▫◫◐◗◎◫▢◫ ◀◫▱◫▽◧◓◞▾▦ ◞◠▦▪◳◧◓◈▾◎▵,I thought you knew where we were going.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Примеры в которых различается порядок и количество цифр

Проверим наличие примеров в которых различается количество и порядок цифр:

In [84]:
def extract_digits(text):
    return ''.join(re.findall(r'\d', text))

df_train['src_digits'] = df_train['src'].apply(extract_digits)
df_train['dst_digits'] = df_train['dst'].apply(extract_digits)

digit_mismatch_df = df_train[df_train['src_digits'] != df_train['dst_digits']]

digit_mismatch_df

/tmp/ipykernel_24846/3825854693.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['src_digits'] = df_train['src'].apply(extract_digits)
/tmp/ipykernel_24846/3825854693.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['dst_digits'] = df_train['dst'].apply(extract_digits)


,src,dst,src_digits,dst_digits
54,"""◝◫▱◎◫▽◂◓◨◎ ◠▷◀◠▻▵ ◟◠◳◬▦▼▪▱◬▨ ◫◒◗▦◪ ◕◫◓◈◗◎▵""","00 , 00:46:50:13 , ""How's the smut business, J...",,0000465013
160,4 ▦▾◎◠◓◠▱▪ ◧◈◠◈◠ ▨◬◓◬▨ ◀◗▱▴◐◗ ◧▱◠▦ ◀◫◓◗ ◚◠◓▵ ▮◪▦◗,We got a sprained ankle in bay four I can put ...,4,
402,9 ◈◫▱ ◀◗▱◗◓■ ▼◠◎ ▩◍▱▴◎▴ ◞◠▦◠▫◬▦◈◠ ▾◞▫◠◈◬◓■ ◠◐▪...,"He can, er, speak nine languages, blow glass, ...",9,
549,◞◗◞▫◪◎◗ ◀◂◳◨▦▼◠ ▥◠◓▻ 2 ▷▪▢◬▦◈◠ ◳◂▱ ◠▱▪▽◧◓◈▾▨▵,We were traveling through the Maxia system.,2,
603,◕◪▱◫▻ ◞▴▦◫ ◠▱◠▼◠▨ ◚▴ ▷◗◉◀◫◓ ▢◠◎◠▦ ▾▦▾▫◠◎◠◳◠▼◠◐...,"I'm going to pick you up at 8:00, show you a n...",,800
...,...,...,...,...
299578,"◝▾ ◀◗◓ ◰►1 ◈◠◚◠◞▪▵""","""I gotta take care of it.",1,
299594,◝▴◒ ◈◨▽▾▦■ ◄◭◈▩◓▵,"Your 5 senses, Boss.",,5
299678,◝▾ ◦▱▱◗◞◂▦ ◆▱◗◍◍◪◓▫ ◎▴◞◪▱▴◞◫▦◈◪ 8▴ ◉◬▨▫▪▦ ◓◪◞◎◪▦▵,This Allison Gliffert thing may have hit an ei...,8,
299954,◦▦▦▴◎ ◀◗◓◗◞◫ ◗◉◗▦ ▨◪◍◗▱ ◂▱◈◨ ◚▴ ◀◗◓ ◕◪▼▴◈◪ ◎◫▱...,She lost 60 million dollars overnight,,60


Удалим найденые битые примеры:

In [85]:
# Оставим только строки в которых количество и порядок цифр в src и dst совпадает
df_train= df_train[df_train['src_digits'] == df_train['dst_digits']]

# Удалим ненужные признаки
df_train = df_train.drop(columns=['src_digits', 'dst_digits'], errors='ignore')


df_train

,src,dst
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
5,◝▴▦ ◫◒▫◪▦ ▨◠◳▫◠◓◠▼◠▨ ◀◗◓◫ ◈▴◐◫▱◈◗◓▵,"For instance, Ben is not a guy who skips work."
...,...,...
299994,▯▴◓▴▽▴ ◕◫▫▫◫◐◗◎◫▢◫ ◀◫▱◫▽◧◓◞▾▦ ◞◠▦▪◳◧◓◈▾◎▵,I thought you knew where we were going.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Примеры в которых различаются символы пунктуации

Проверим наличие примеров в которых в признаках различаются символы пунктуации:

In [86]:
punctuation_string = ']"%;@\'&?!+:)[(/'

df_train['src_punctuation'] = df_train['src'].apply(lambda x: ''.join([char for char in x if char in punctuation_string]))
df_train['dst_punctuation'] = df_train['dst'].apply(lambda x: ''.join([char for char in x if char in punctuation_string]))

punctuation_mismatch_train_df = df_train[df_train['src_punctuation'] != df_train['dst_punctuation']]
punctuation_mismatch_train_df

,src,dst,src_punctuation,dst_punctuation
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?,?,'?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,...",,''
12,◱▵▵▵◎◧◓ ◉◨▱▱◨▨ ◧▱◨◓◈◨▵,She-- she's the purple sandpiper.,,'
13,◝◫◓ ◠◓▨◠◈◠◒▪◎▱◠ ◀◨▱◨◒◠▼◠◐▪◎▵,I'm meeting a friend.,,'
17,○◎◠ 12 ▨◗◒◗▱▴◓▵ ► ◙▴◓▫ ◪▫◎◪▵,But there's 12 of them.,,'
...,...,...,...,...
299984,◄◠◓▨'◬▦ ▷◠▼▨▴◓ ◠◓▨◠◈◠◒◬▦◬ ◀◫▱◫▽◧◓◞◨▦▵▵▵,Very funny (!,',(!
299992,◫▦◞◠▦▱◠◓▪▦ ◀◫▱◎▴◞◗▦◗▦ ◣▦◪◎▱◫ ◂▱◈▾◐▾▦◨ ◈▩◒▩▦▩◳◂...,I just think it's important that people know.,,'
299993,▭◪▽■ ◧ ▴▻◪▽◈◫◓ ◀◨◓◠◳◠ ◕▴▱◎◗▽◂◓▵,"I... Hey, she hasn't been over here in a while.",,'
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil...",,'


Удалим из тренировочной части все приемры в которых пунктуация различается в признаках:

In [87]:
# Удалим из тренировочной части все примеры, в которых пунктуация различается в признаках
df_train = df_train[df_train['src_punctuation'] == df_train['dst_punctuation']]

# Удалим ненужные признаки
df_train = df_train.drop(columns=['src_punctuation', 'dst_punctuation'], errors='ignore')

df_train

,src,dst
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
5,◝▴▦ ◫◒▫◪▦ ▨◠◳▫◠◓◠▼◠▨ ◀◗◓◫ ◈▴◐◫▱◈◗◓▵,"For instance, Ben is not a guy who skips work."
6,▭◂◒◉◠ ▨◠▱ ◡◠▨◂◀▵,"Goodbye, Jakob."
7,◝◨◓◠◈◠▨◗ ▷◪◓▨◪◞◫ ◳◠▽◠ ◀◬◓◠▨▪◓▵,Sure beats the foam domes round here.
...,...,...
299989,◀▾ ◗▨◗ ◠◈◠◎▪▦ ◉▪▻▱◠▨ ▽▩▢◭◒◭▦▩ ◫▢▱◪◎▴▨ ◈▪◒◬▦◈◠ ...,"Yes, I will do anything besides watch these tw..."
299990,◝◠▦◠ ◈◠ 500 ◈◂▱◠◓ ◣▦◪◓◈◫▵,"He tapped me up for $500, too."
299994,▯▴◓▴▽▴ ◕◫▫▫◫◐◗◎◫▢◫ ◀◫▱◫▽◧◓◞▾▦ ◞◠▦▪◳◧◓◈▾◎▵,I thought you knew where we were going.
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.


### Дедубликация примеров

Проверим наличие дубликатов в примерах по признаку `src`:

In [88]:
duplicates = df_train[df_train.duplicated(subset=['src'], keep=False)]
duplicates

,src,dst
102,▯▴◈▴▦ ◀◨◓◠◳◠ ◕◪▱◈◫▦?,why did you come here?
103,▯▴◈▴▦ ◀◨◓◠◳◠ ◕◪▱◈◫▦?,Why did you come?
104,▯▴◈▴▦ ◀◨◓◠◳◠ ◕◪▱◈◫▦?,Why have you come here?
105,▯▴◈▴▦ ◀◨◓◠◳◠ ◕◪▱◈◫▦?,Why did you come here?
107,▯▴◈▴▦ ◀◨◓◠◳◠ ◕◪▱◈◫▦?,Why did you come?
...,...,...
299822,◝◪▦◫ ◓◠▷◠▫◞▪▢ ▴◈◗◳◧◓▵,It just makes me uncomfortable.
299866,◆▴▱◈◫◐◫▦ ◫◉◗▦ ▫▴◒▴▨▨▩◓▱◪◓▵,Mm. Thanks for coming.
299867,◆▴▱◈◫◐◫▦ ◫◉◗▦ ▫▴◒▴▨▨▩◓▱◪◓▵,Thanks for coming.
299878,▭◠▽◈◫ ◂◐▱▾◎!,"Come, Whitey!"


Удалим все дубликаты оставляя первый встреченный пример:

In [89]:
df_train = df_train.drop_duplicates(subset=['src'], keep='first')
df_train

,src,dst
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
5,◝▴▦ ◫◒▫◪▦ ▨◠◳▫◠◓◠▼◠▨ ◀◗◓◫ ◈▴◐◫▱◈◗◓▵,"For instance, Ben is not a guy who skips work."
6,▭◂◒◉◠ ▨◠▱ ◡◠▨◂◀▵,"Goodbye, Jakob."
7,◝◨◓◠◈◠▨◗ ▷◪◓▨◪◞◫ ◳◠▽◠ ◀◬◓◠▨▪◓▵,Sure beats the foam domes round here.
...,...,...
299989,◀▾ ◗▨◗ ◠◈◠◎▪▦ ◉▪▻▱◠▨ ▽▩▢◭◒◭▦▩ ◫▢▱◪◎▴▨ ◈▪◒◬▦◈◠ ...,"Yes, I will do anything besides watch these tw..."
299990,◝◠▦◠ ◈◠ 500 ◈◂▱◠◓ ◣▦◪◓◈◫▵,"He tapped me up for $500, too."
299994,▯▴◓▴▽▴ ◕◫▫▫◫◐◗◎◫▢◫ ◀◫▱◫▽◧◓◞▾▦ ◞◠▦▪◳◧◓◈▾◎▵,I thought you knew where we were going.
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.


Проверим дубликаты по признаку `dst`:

In [ ]:
# duplicates = df_train[df_train.duplicated(subset=['dst'], keep=False)]
# duplicates

## Сохрание очищеной тренировочной части датасета

In [90]:
# Обновление датасета после фильтрации
dataset['train'] = Dataset.from_pandas(df_train, preserve_index=False)

# Проверка количества строк после фильтрации
print(f"Number of rows after filtering in train split: {dataset['train'].shape[0]}")


dataset['train'].to_json('train.jsonl', orient='records', lines=True, index=False, force_ascii=False)

Number of rows after filtering in train split: 116456


Creating json from Arrow format:   0%|          | 0/117 [00:00<?, ?ba/s]

18379446